In [1]:
import pandas as pd
from tqdm import tqdm


In [2]:
from dataclasses import dataclass
from typing import Optional, List
import pandas as pd


# -----------------------------
# Body Part
# -----------------------------

@dataclass
class BodyPoint:
    x: float
    y: float
    z: float


# -----------------------------
# Player Skeleton (21 joints)
# -----------------------------

@dataclass
class PlayerSkeleton:

    jersey_number: int
    team: int

    left_ear: Optional[BodyPoint] = None
    nose: Optional[BodyPoint] = None
    right_ear: Optional[BodyPoint] = None

    left_shoulder: Optional[BodyPoint] = None
    neck: Optional[BodyPoint] = None
    right_shoulder: Optional[BodyPoint] = None

    left_elbow: Optional[BodyPoint] = None
    right_elbow: Optional[BodyPoint] = None

    left_wrist: Optional[BodyPoint] = None
    right_wrist: Optional[BodyPoint] = None

    left_hip: Optional[BodyPoint] = None
    pelvis: Optional[BodyPoint] = None
    right_hip: Optional[BodyPoint] = None

    left_knee: Optional[BodyPoint] = None
    right_knee: Optional[BodyPoint] = None

    left_ankle: Optional[BodyPoint] = None
    right_ankle: Optional[BodyPoint] = None

    left_heel: Optional[BodyPoint] = None
    left_toe: Optional[BodyPoint] = None

    right_heel: Optional[BodyPoint] = None
    right_toe: Optional[BodyPoint] = None


# -----------------------------
# Ball
# -----------------------------

@dataclass
class Ball:
    position_x: float
    position_y: float
    position_z: float
    velocity_x: float
    velocity_y: float
    velocity_z: float


# -----------------------------
# Frame
# -----------------------------

@dataclass
class Frame:
    frame_number: int
    version: int
    type: int
    skeleton_count: int
    ball_exists: bool
    skeletons: List[PlayerSkeleton]
    ball: Optional[Ball]


# -----------------------------
# Mapping from ID → attribute
# -----------------------------

PART_ID_MAP = {
    1: "left_ear",
    2: "nose",
    3: "right_ear",
    4: "left_shoulder",
    5: "neck",
    6: "right_shoulder",
    7: "left_elbow",
    8: "right_elbow",
    9: "left_wrist",
    10: "right_wrist",
    11: "left_hip",
    12: "pelvis",
    13: "right_hip",
    14: "left_knee",
    15: "right_knee",
    16: "left_ankle",
    17: "right_ankle",
    18: "left_heel",
    19: "left_toe",
    20: "right_heel",
    21: "right_toe",
}


# -----------------------------
# Parsing functions
# -----------------------------

def parse_player(skeleton_dict):

    player = PlayerSkeleton(
        jersey_number=skeleton_dict["jersey_number"],
        team=skeleton_dict.get("team", -1),
    )

    for part in skeleton_dict["parts"]:

        part_id = part["name"]

        if part_id not in PART_ID_MAP:
            continue

        attr_name = PART_ID_MAP[part_id]

        point = BodyPoint(
            x=part["position_x"],
            y=part["position_y"],
            z=part["position_z"],
        )

        setattr(player, attr_name, point)

    return player


def parse_ball(ball_dict):

    return Ball(
        position_x=ball_dict["position_x"],
        position_y=ball_dict["position_y"],
        position_z=ball_dict["position_z"],
        velocity_x=ball_dict["velocity_x"],
        velocity_y=ball_dict["velocity_y"],
        velocity_z=ball_dict["velocity_z"],
    )


def parse_frame(row):

    skeletons = [parse_player(s) for s in row["skeletons"]]

    ball = None
    if row["ball_exists"] and row["ball"] is not None:
        ball = parse_ball(row["ball"])

    return Frame(
        frame_number=row["frame_number"],
        version=row["version"],
        type=row["type"],
        skeleton_count=row["skeleton_count"],
        ball_exists=row["ball_exists"],
        skeletons=skeletons,
        ball=ball,
    )


# -----------------------------
# Load parquet
# -----------------------------

def load_frames(parquet_path):

    df = pd.read_parquet(parquet_path)

    frames = []

    for _, row in tqdm(df.iterrows()):
        frames.append(parse_frame(row))

    return frames

In [ ]:
frames = load_frames("anonymized-limbtracking.parquet")

print("Frames loaded:", len(frames))


In [ ]:
first_frame = frames[0]
first_player = first_frame.skeletons[0]

print("Frame number:", first_frame.frame_number)
print("Player jersey:", first_player.jersey_number)

if first_player.left_ankle:
    print("Left ankle X:", first_player.left_ankle.x)

if first_frame.ball:
    print("Ball position:", first_frame.ball.position_x, first_frame.ball.position_y)